# Soft Robotic Arm — Controller: Delay-Aware NDOSMC on the Identified M2 Model

This notebook is the project's final submitted controller. It uses:

- **The M2 model** — the two first-order equations identified from
  input/output data (`scripts/task_space_model.py` in the project repo), *not*
  the physics-derived reduced model used in the other notebooks here. M2 was
  validated open-loop against three held-out experiments never used in fitting
  and sits close to the simulated sensor noise floor (0.10mm), well ahead of the
  simpler M1 model — see the model-validation discussion for the full comparison.
  The two equations:

  $$\dot{y}_i = -a_i y_i + a_i (Hp)_i \qquad \text{(mechanics: pressure} \to \text{tip position)}$$
  $$\tau\,\dot{p}_j = g_j\, u_j(t - T) - p_j \qquad \text{(actuator: command} \to \text{pressure, with dead time } T \text{ and lag } \tau\text{)}$$

- **The controller** — a Sliding Mode Controller with a Nonlinear Disturbance
  Observer (NDOSMC), per Qiao, Tao & Zhang (2024), made *delay-aware*: it runs
  an internal delay-free copy of the model (a Smith-predictor-style trick) so
  the control law reacts to where the tip will be once the actuator's 0.5s dead
  time has passed, and it inverts the actuator's 0.6s lag when forming the
  command. This is the `DelayAwareNDOSMC` controller from `scripts/ndosmc.py`,
  reproduced here with the exact tuned gains from `scripts/compare_controllers.py`
  (`lam=4.0, l1=0.1, l2=0.0, mu=1.0, Phi=1.0, eps=1.0`).

**Why SMC over MPC on this model.** Both reach essentially the same whole-run
accuracy on M2 (SMC 1.21mm vs. MPC's reproduced 1.22mm — not a meaningful
difference), but SMC needs far less machinery (no Kalman filter, no online
prediction/preview tuning) and its numbers reproduce exactly from the shared
code. MPC's own reported "overall" number did not reproduce exactly when we
re-ran it, so SMC is the safer, equally-accurate choice for a submission that
has to be independently re-run and checked.

All model parameters below are the exact values the identification
script fit (copied from `docs/figures/task_space_model.json`), embedded
directly so this notebook does not depend on any external file — it only
needs the published `soft-robotic-arm` simulator package.

## 1 — Install and set up (same as the other notebooks)

In [ ]:
%pip install -q --upgrade --no-cache-dir soft-robotic-arm

In [ ]:
import os, sys
os.environ["MUJOCO_GL"] = "egl"
os.environ["PYOPENGL_PLATFORM"] = "egl"
for module_name in tuple(sys.modules):
    if module_name == "soft_robotic_arm" or module_name.startswith("soft_robotic_arm."):
        del sys.modules[module_name]

from collections import deque
from dataclasses import dataclass, field

import numpy as np
import matplotlib.pyplot as plt
import mujoco
import soft_robotic_arm
from soft_robotic_arm import make_calibrated_sim
from soft_robotic_arm.evaluation import CircularTrackingTask, evaluate_controller

CONTROL_HZ = 100.0
BASELINE_PSI = 4.5
MAX_PSI = 9.0
print("Simulator ready")

## 2 — Identified M2 model (system identification)

Embedded directly from `docs/figures/task_space_model.json` so this notebook
is self-contained — no external model file to carry around at submission
time. Three pieces, all fit from data (none of these numbers come from the
robot's physical dimensions or material properties):

- **`B` (= `G`)** — the static 3×3 command-deviation &rarr; virtual-input gain
  matrix. It is only needed by the shared base-controller class below to size
  its (unused, for this controller) inverse; the actual control law uses
  `B_measured` instead (see next point).
- **`B_measured` (= `H`)** and **`axes_measured`** — the M2 mechanics: `H` maps
  measured-pressure deviations to tip-position response, and each axis's `a`
  (how fast that axis's position error closes) is stored as a tiny first-order
  `AxisModel` (`k=1, c=1/a`) so it plugs into the same controller code used for
  the paper's scheduled (M1) model.
- **`actuator`** — gain, dead time (`0.5s`), and lag (`0.6s`) per channel,
  i.e. Equation 2 above.

In [ ]:
@dataclass
class AxisModel:
    """Per-axis c*X' + k*X = v, i.e. alpha = k/c, beta = 1/c."""
    k_poly: tuple
    c_poly: tuple
    rho_range: tuple = (-np.inf, np.inf)
    k_spread: float = 0.0
    c_spread: float = 0.0

    @classmethod
    def unit_dc(cls, a: float) -> "AxisModel":
        """X' = -a X + a v (M2's plain first-order axis): k = 1, c = 1/a."""
        return cls(k_poly=(0.0, 0.0, 1.0), c_poly=(0.0, 0.0, 1.0 / a))

    def kc(self, rho: float):
        r = float(np.clip(rho, *self.rho_range))
        return float(np.polyval(self.k_poly, r)), float(np.polyval(self.c_poly, r))

    def params(self, rho: float):
        k, c = self.kc(rho)
        return k / c, 1.0 / c


@dataclass
class ActuatorModel:
    """Command deviation -> measured pressure deviation per segment: gain * lag(u(t - T))."""
    gain: np.ndarray
    delay_s: float
    tau_s: float


@dataclass
class TaskSpaceModel:
    """Only the fields DelayAwareNDOSMC actually uses: B (for the shared base class),
    B_measured + axes_measured (the M2 mechanics law), and actuator (Equation 2)."""
    B: np.ndarray
    B_measured: np.ndarray
    axes_measured: list
    actuator: ActuatorModel
    axes: list = field(default_factory=list)  # unused by DelayAwareNDOSMC; kept for API parity


# --- exact fitted values, from docs/figures/task_space_model.json ---

G = np.array([
    [0.00031357166002859207, -1.9827452731012267,  -0.00202631055327337],
    [2.0256520920838583,      0.0008803893480654749, -2.1062044574564767],
    [-0.5509286525717835,    -0.48416201174123447,   -0.5697743374585689],
])

ACTUATOR = ActuatorModel(
    gain=np.array([1.1198163557756944, 1.0963542905388663, 1.1640064989851946]),
    delay_s=float(np.mean([0.5, 0.5, 0.5])),                                   # 0.5 s dead time
    tau_s=float(np.mean([0.5995934524149064, 0.5992802573570557, 0.5995415244188516])),  # ~0.6 s lag
)

M2_MECHANICS = {
    "x": dict(a=4.768630722403884,
              H=[-0.00328980834291186, -1.81213634611892,    -0.0019863480130171236]),
    "y": dict(a=4.699171384202557,
              H=[1.8230768645800255,  -0.0017432863449094197, -1.8133464514018007]),
    "z": dict(a=15.493503802453375,
              H=[-0.49285145163275845, -0.4460844286442736,   -0.4895769294975046]),
}

MODEL = TaskSpaceModel(
    B=G,
    B_measured=np.array([M2_MECHANICS[ax]["H"] for ax in ("x", "y", "z")]),
    axes_measured=[AxisModel.unit_dc(M2_MECHANICS[ax]["a"]) for ax in ("x", "y", "z")],
    actuator=ACTUATOR,
)
print("a (x, y, z):", [M2_MECHANICS[ax]["a"] for ax in ("x", "y", "z")])
print("actuator dead time / lag (s):", MODEL.actuator.delay_s, "/", round(MODEL.actuator.tau_s, 3))

## 3 — Controller: Delay-Aware NDOSMC

Three pieces, matching `scripts/ndosmc.py`:

1. **`_Base`** — shared bookkeeping (home pose, numerically-differentiated
   reference velocity/acceleration, mm scaling).
2. **`NDOB`** — the nonlinear disturbance observer (paper eqs. 6–9): estimates
   a lumped disturbance `d_hat` per axis from the tracking error, so the
   control law can cancel unmodelled effects instead of just reacting to them.
3. **`PaperNDOSMC`** — the sliding-mode law itself (paper eq. 12 with the
   adaptive switching gain of eq. 14): drives the sliding variable
   `sigma = position error` to zero via equivalent control plus a saturated
   switching term, with `d_hat` subtracted out.
4. **`DelayAwareNDOSMC`** (extends `PaperNDOSMC`) — the actual controller run
   below. Two additions on top of plain NDOSMC:
   - **Smith-predictor-style delay compensation**: it keeps an internal,
     delay-free copy of the M2 model running alongside the real plant. The
     difference between that model's current state and its state one dead-time
     ago is added to the measurement, so the control law reacts to an estimate
     of the tip position *T seconds from now* rather than to the measurement as
     it stands — the actuator's dead time no longer blindsides the controller.
   - **Actuator-lag inversion**: instead of commanding the desired pressure
     directly, it commands `p_desired + tau * d(p_desired)/dt`, which is exactly
     the pressure needed so that, once it passes through the actuator's own lag
     (Equation 2), the *result* is `p_desired` right away.

   Gains used below (`lam=4.0, l1=0.1, l2=0.0, mu=1.0, Phi=1.0, eps=1.0`) are
   the tuned "Delay-aware NDOSMC (M2)" configuration from
   `scripts/compare_controllers.py` — the best whole-run RMSE (1.21mm) found
   for this controller/model pair.

In [ ]:
class _Base:
    """Shared bookkeeping: home pose, reference derivatives, mm scaling."""

    def __init__(self, model: TaskSpaceModel, lam: float):
        self.m = model
        self.lam = lam
        self.B_inv = np.linalg.inv(model.B)   # unused by DelayAwareNDOSMC's own compute(); kept for parity
        self.home = None
        self.t0 = None
        self._ref_hist = deque(maxlen=3)
        self._t_hist = deque(maxlen=3)
        self.dt = 1.0 / CONTROL_HZ

    def _frame(self, t, obs, ref):
        if self.home is None:
            self.home = np.asarray(obs["tip_pos"], float).copy()
            self.t0 = t
        q = 1000.0 * (np.asarray(obs["tip_pos"], float) - self.home)      # mm
        qd_dot_meas = 1000.0 * np.asarray(obs["tip_vel"], float)          # mm/s
        r = 1000.0 * (np.asarray(ref, float) - self.home)
        self._ref_hist.append(r.copy())
        self._t_hist.append(t)
        r_dot = np.zeros(3)
        r_ddot = np.zeros(3)
        if len(self._t_hist) >= 2:
            r_dot = (self._ref_hist[-1] - self._ref_hist[-2]) / self.dt
        if len(self._t_hist) == 3:
            r_dot_prev = (self._ref_hist[-2] - self._ref_hist[-3]) / self.dt
            r_ddot = (r_dot - r_dot_prev) / self.dt
        return q, qd_dot_meas, r, r_dot, r_ddot


class NDOB:
    """Nonlinear disturbance observer, paper eqs. 6-9, forward Euler at the control rate."""

    def __init__(self, l1, l2, dt):
        self.l1, self.l2, self.dt = l1, l2, dt
        self.z = None

    def p(self, q):
        return self.l1 * q + self.l2 * q**2 * np.sign(q)

    def gain(self, q):
        return self.l1 + 2.0 * self.l2 * np.abs(q)

    def estimate(self, q):
        if self.z is None:
            self.z = -self.p(q)  # d_hat(0) = 0
        return self.z + self.p(q)

    def update(self, q, alpha, beta_v):
        """z' = g(-a q - b u - d_hat), a = -alpha, b u = beta v."""
        d_hat = self.estimate(q)
        self.z = self.z + self.dt * self.gain(q) * (alpha * q - beta_v - d_hat)

In [ ]:
class PaperNDOSMC(_Base):
    """NDOSMC, paper eq. 12 with the adaptive gain of eq. 14 and a sat boundary layer mu."""

    def __init__(self, model, lam=2.0, l1=5.0, l2=0.0, Phi=1e-3, eps=1e-3, mu=1e-3, observer=True):
        super().__init__(model, lam)
        self.obs = NDOB(l1, l2, self.dt)
        self.use_observer = observer
        self.Phi, self.eps, self.mu = Phi, eps, mu

    def _law(self, q, r, r_dot, rho, d_hat, t_rel, axes):
        g = self.obs.gain(q)
        v = np.zeros(3)
        alpha = np.zeros(3)
        beta = np.zeros(3)
        for i, ax in enumerate(axes):
            a, b = ax.params(rho[i])
            alpha[i], beta[i] = a, b
            sigma = q[i] - r[i]
            eta = self.Phi * np.exp(-g[i] * t_rel) + self.eps / g[i] if self.use_observer else self.eps
            sat = np.clip(sigma / self.mu, -1.0, 1.0)
            v[i] = (r_dot[i] + a * q[i] - self.lam * sigma - eta * sat - d_hat[i]) / b
        return v, alpha, beta

In [ ]:
class DelayAwareNDOSMC(PaperNDOSMC):
    """NDOSMC on a Smith-predicted state, with the identified actuator delay and lag inverted.

    Mechanics are M2 (B_measured, axes_measured), driven by MEASURED pressure. The outer
    law asks for an effective pressure; the command inverts the actuator lag. An internal
    delay-free copy of the model is advanced every step so the outer loop sees an estimate
    of the tip position one dead-time ahead of the raw measurement.
    """

    def __init__(self, model: TaskSpaceModel, exact_preview: bool = False, **kw):
        super().__init__(model, **kw)
        self.exact_preview = exact_preview
        act = model.actuator
        self.act = act
        self.Bp = model.B_measured
        self.Bp_inv = np.linalg.inv(self.Bp)
        self.n_delay = max(1, int(round(act.delay_s * CONTROL_HZ)))
        self.p_model = np.zeros(3)
        self.x_model = np.zeros(3)
        self.x_hist = deque([np.zeros(3)] * self.n_delay, maxlen=self.n_delay)
        self.p_des_prev = None

    def compute(self, t, obs, ref):
        q, _, r, r_dot, r_ddot = self._frame(t, obs, ref)
        T = self.n_delay * self.dt
        q_pred = q + self.x_model - self.x_hist[0]
        if self.exact_preview:
            r_pred, r_dot_pred = r, r_dot
        else:
            r_pred = r + T * r_dot + 0.5 * T**2 * r_ddot
            r_dot_pred = r_dot + T * r_ddot
        rho = np.zeros(3)  # M2 mechanics are not scheduled
        d_hat = self.obs.estimate(q_pred) if self.use_observer else np.zeros(3)
        v, alpha, beta = self._law(q_pred, r_pred, r_dot_pred, rho, d_hat, t - self.t0,
                                   self.m.axes_measured)
        p_des = self.Bp_inv @ v
        p_des_dot = np.zeros(3) if self.p_des_prev is None else (p_des - self.p_des_prev) / self.dt
        self.p_des_prev = p_des
        p_cmd_eff = p_des + self.act.tau_s * p_des_dot        # lag inverse
        u_cmd = BASELINE_PSI + p_cmd_eff / self.act.gain
        cmd = np.clip(u_cmd, 0.0, MAX_PSI)

        # advance the internal delay-free model with the command actually sent
        p_target = self.act.gain * (cmd - BASELINE_PSI)
        self.p_model = self.p_model + self.dt / self.act.tau_s * (p_target - self.p_model)
        v_model = self.Bp @ self.p_model
        for i, ax in enumerate(self.m.axes_measured):
            a, b = ax.params(rho[i])
            self.x_model[i] += self.dt * (-a * self.x_model[i] + b * v_model[i])
        self.x_hist.append(self.x_model.copy())
        self.obs.update(q_pred, alpha, beta * v_model)
        return cmd

## 4 — Evaluate on the scored task

Same `CircularTrackingTask()` (7mm radius, 0.1Hz, 5s settle + 30s tracked) and
`make_calibrated_sim(seed=0)` harness used for every other controller in this
project, so the number below is directly comparable.

In [ ]:
task = CircularTrackingTask()
sim = make_calibrated_sim(control_hz=CONTROL_HZ, seed=0)
controller = DelayAwareNDOSMC(MODEL, lam=4.0, l1=0.1, l2=0.0, mu=1.0, Phi=1.0, eps=1.0)

result = evaluate_controller(controller, task=task, sim=sim)

print(f"Task: {task.radius_m*1000:.0f}mm radius circle @ {task.frequency_hz}Hz, "
      f"{task.track_s:.0f}s tracked after {task.settle_s:.0f}s settle")
print(f"RMSE (whole run):     {result.rmse_mm:.2f} mm")
print(f"Max error:            {result.max_error_mm:.2f} mm")
print(f"Phase lag:            {result.phase_lag_s:.3f} s")
print(f"Effort:               {result.effort_psi:.3f} psi (RMS deviation from baseline)")

tail_sel = result.times >= 5.0
tail_err = np.linalg.norm(result.tips[tail_sel, :2] - result.references[tail_sel, :2], axis=1) * 1000.0
print(f"RMSE (after 5s, steady state): {np.sqrt(np.mean(tail_err**2)):.3f} mm")

fig, axes = result.plot(label="Delay-aware NDOSMC (M2)")
plt.show()

## 5 — Held-out task (honesty check)

The gains above were picked by sweeping on the *scored* task itself, so a
held-out trajectory — a larger, faster circle, different random seed — is run
too. It is **not** expected to match the scored number; it just confirms the
controller doesn't fall apart off the exact tuned trajectory.

In [ ]:
held_out_task = CircularTrackingTask(radius_m=0.010, frequency_hz=0.15)
held_out_sim = make_calibrated_sim(control_hz=held_out_task.control_hz, seed=5)
held_out_controller = DelayAwareNDOSMC(MODEL, lam=4.0, l1=0.1, l2=0.0, mu=1.0, Phi=1.0, eps=1.0)

held_out_result = evaluate_controller(held_out_controller, task=held_out_task, sim=held_out_sim)
print(f"Task: {held_out_task.radius_m*1000:.0f}mm radius circle @ {held_out_task.frequency_hz}Hz, seed 5")
print(f"RMSE (whole run): {held_out_result.rmse_mm:.2f} mm")
print(f"Max error:        {held_out_result.max_error_mm:.2f} mm")
print(f"Effort:           {held_out_result.effort_psi:.3f} psi")

## 6 — Random, irregular trajectories (trajectory known in advance)

The professor's final test will use random trajectories that are not very regular, and it is
fair to assume the trajectory is known in advance. Two consequences:

- **Test:** two families of random reference, three seeds each, 30 s, same plant and seed as
  the scored task: (a) **smooth random**: per axis a sum of six sines with random
  frequencies (0.03–0.3 Hz), phases and weights, scaled to ±6 mm; never repeats; (b) **random
  waypoints**: straight lines between random points (±6 mm), 1–2.5 s per segment, so the
  path has sharp corners at irregular times. Both start at the settled tip.
- **Controller:** every command reaches the arm `T = 0.5 s` late, so the controller must aim
  at where the reference *will be*, `r(t + T)`. Section 4's controller only receives `r(t)`
  and estimates `r(t + T)` with a Taylor series, `r + T·ṙ + ½T²·r̈` (`exact_preview=False`).
  When the trajectory is known, it can read `r(t + T)` directly (`exact_preview=True`); the
  `KnownTrajectory` wrapper below does that. The score is still against `r(t)`.


In [ ]:
DT = 1.0 / CONTROL_HZ
TRACK_S = 30.0          # tracked seconds, as in CircularTrackingTask
SETTLE_S = 5.0          # settle at the baseline first, as evaluate_controller does
HORIZON_S = 0.5         # dead time T: how far ahead the known trajectory is read


def smooth_random(seed, span_mm=6.0, f_lo=0.03, f_hi=0.3, n=6):
    """Offset from home (m) as a function of tracking time: per axis, a sum of n sines with
    random frequencies in [f_lo, f_hi] Hz, phases and weights, scaled to +-span_mm."""
    g = np.random.default_rng(seed)
    f = g.uniform(f_lo, f_hi, (2, n)); ph = g.uniform(0, 2 * np.pi, (2, n)); a = g.uniform(0.3, 1.0, (2, n))
    raw = lambda t: (a * np.sin(2 * np.pi * f * np.asarray(t)[..., None, None] + ph)).sum(-1)
    t = np.arange(0.0, TRACK_S, DT)
    scale = span_mm / 1000.0 / np.abs(raw(t) - raw(0.0)).max(axis=0)
    return lambda tau: np.append((raw(tau) - raw(0.0)) * scale, 0.0)


def random_waypoints(seed, span_mm=6.0, seg_lo=1.0, seg_hi=2.5):
    """Offset from home (m): straight lines between random waypoints in +-span_mm, with random
    segment durations in [seg_lo, seg_hi] s; starts at home."""
    g = np.random.default_rng(seed)
    times, pts = [0.0], [np.zeros(2)]
    while times[-1] < TRACK_S + 2 * HORIZON_S:
        times.append(times[-1] + g.uniform(seg_lo, seg_hi))
        pts.append(g.uniform(-span_mm, span_mm, 2) / 1000.0)
    pts = np.array(pts)
    return lambda tau: np.array([np.interp(tau, times, pts[:, 0]), np.interp(tau, times, pts[:, 1]), 0.0])


class KnownTrajectory:
    """Hands the controller the reference HORIZON_S ahead, read from the known trajectory.
    The wrapped controller must use exact_preview=True (it then takes `ref` as r(t + T))."""

    def __init__(self, controller, traj):
        self.c, self.traj, self.t0, self.home = controller, traj, None, None

    def compute(self, t, obs, ref):
        if self.t0 is None:
            self.t0 = t
            self.home = np.asarray(ref, float) - self.traj(0.0)
        return self.c.compute(t, obs, self.home + self.traj(t - self.t0 + HORIZON_S))


def track(controller, traj, seed=0):
    """evaluate_controller's loop for any reference: settle at 4.5 psi, home = settled tip,
    then each step command -> step; the tip after the step is scored against the reference
    given at that step. Returns tips, references (m) and commands (psi)."""
    sim = make_calibrated_sim(control_hz=CONTROL_HZ, seed=seed)
    obs = sim.reset()
    for _ in range(int(round(SETTLE_S * CONTROL_HZ))):
        obs = sim.step(np.full(3, BASELINE_PSI))
    home = np.asarray(obs["tip_pos"], float).copy()
    n = int(round(TRACK_S * CONTROL_HZ))
    tips, refs, cmds = np.empty((n, 3)), np.empty((n, 3)), np.empty((n, 3))
    for k in range(n):
        ref = home + traj(k * DT)
        cmd = np.clip(np.asarray(controller.compute(float(obs["time"]), obs, ref), float), 0.0, MAX_PSI)
        obs = sim.step(cmd)
        tips[k], refs[k], cmds[k] = obs["tip_pos"], ref, cmd
    sim.close()
    return tips, refs, cmds


def scores(tips, refs, cmds):
    e = 1000.0 * np.linalg.norm(tips[:, :2] - refs[:, :2], axis=1)
    return dict(rmse=np.sqrt(np.mean(e**2)), rmse_after_5s=np.sqrt(np.mean(e[int(5 * CONTROL_HZ):] ** 2)),
                max_after_5s=e[int(5 * CONTROL_HZ):].max(), sat=100 * np.mean((cmds <= 0) | (cmds >= MAX_PSI)))


# Check: track() reproduces evaluate_controller on the scored circle (1.21 mm).
from soft_robotic_arm.evaluation import make_reference
_circle = lambda tau: make_reference(np.zeros(3), tau, CircularTrackingTask())
_s = scores(*track(DelayAwareNDOSMC(MODEL, lam=4.0, l1=0.1, l2=0.0, mu=1.0, Phi=1.0, eps=1.0), _circle))
print(f"scored circle through track(): RMSE {_s['rmse']:.2f} mm (evaluate_controller: {result.rmse_mm:.2f} mm)")

In [ ]:
GAINS = dict(lam=4.0, l1=0.1, l2=0.0, mu=1.0, Phi=1.0, eps=1.0)
CONTROLLERS = {
    "estimated r(t+T) (Taylor)": lambda traj: DelayAwareNDOSMC(MODEL, **GAINS),
    "known trajectory": lambda traj: KnownTrajectory(DelayAwareNDOSMC(MODEL, exact_preview=True, **GAINS), traj),
}
FAMILIES = {"smooth random": smooth_random, "random waypoints": random_waypoints}
SEEDS = (101, 102, 103)

runs = {}
print(f"{'trajectory':<18}{'seed':>5}  {'controller':<27}{'RMSE':>7}{'after 5 s':>11}{'max after 5 s':>15}{'sat %':>7}")
for fam, gen in FAMILIES.items():
    for seed in SEEDS:
        traj = gen(seed)
        for name, make in CONTROLLERS.items():
            tips, refs, cmds = track(make(traj), traj)
            runs[(fam, seed, name)] = (tips, refs)
            s = scores(tips, refs, cmds)
            print(f"{fam:<18}{seed:>5}  {name:<27}{s['rmse']:>7.2f}{s['rmse_after_5s']:>11.2f}"
                  f"{s['max_after_5s']:>15.2f}{s['sat']:>7.1f}")

In [ ]:
# One example per family. Colour-blind-safe: black dashed reference; the two controllers differ
# in colour (Okabe-Ito orange / blue), line style and marker.
STYLE = {"estimated r(t+T) (Taylor)": dict(color="#E69F00", ls="--", marker="^", markevery=150),
         "known trajectory": dict(color="#0072B2", ls="-", marker="o", markevery=150)}
fig, axs = plt.subplots(2, 2, figsize=(13, 10))
for row, fam in enumerate(FAMILIES):
    t = np.arange(int(TRACK_S * CONTROL_HZ)) * DT
    for name in CONTROLLERS:
        tips, refs = runs[(fam, SEEDS[0], name)]
        axs[row, 0].plot(1000 * (tips[:, 0] - refs[0, 0]), 1000 * (tips[:, 1] - refs[0, 1]), lw=1.8, label=name, **STYLE[name])
        axs[row, 1].plot(t, 1000 * np.linalg.norm(tips[:, :2] - refs[:, :2], axis=1), lw=1.8, label=name, **STYLE[name])
    axs[row, 0].plot(1000 * (refs[:, 0] - refs[0, 0]), 1000 * (refs[:, 1] - refs[0, 1]), "k--", lw=2.5, label="reference")
    axs[row, 0].set(title=f"{fam}, seed {SEEDS[0]}: tip path", xlabel="x − home (mm)", ylabel="y − home (mm)")
    axs[row, 0].set_aspect("equal", adjustable="datalim")
    axs[row, 1].set(title=f"{fam}, seed {SEEDS[0]}: planar error", xlabel="tracking time (s)", ylabel="error (mm)")
    axs[row, 1].set_ylim(0, None)
    for ax in axs[row]:
        ax.grid(alpha=0.3); ax.legend(fontsize=11)
fig.tight_layout()
plt.show()

**Reading the result.** Knowing the trajectory removes most of the error on irregular paths:
the Taylor estimate extrapolates the current velocity and acceleration 0.5 s ahead, so it
overshoots every corner of the waypoint paths and every fast turn of the smooth ones, while
the known trajectory gives the true `r(t + T)`. Same controller, same gains; only the
reference handed to it changes. Whole-run RMSE includes the first 0.5–1 s, before any command
can reach the arm.

## 7 — Summary and honest limitations

| | Scored task (7mm, 0.1Hz, seed 0) | Held-out task (10mm, 0.15Hz, seed 5) |
|---|---:|---:|
| RMSE (whole run) | **1.21 mm** | 1.97 mm |
| RMSE (after 5s, steady state) | 0.10 mm | 0.95 mm |
| Max error | 7.00 mm | 10.00 mm |
| Effort (psi) | 2.15 | 3.31 |
| Commands saturated | 0.04% | 21.2% |

**Takeaway.** On the task the project is scored on, this controller tracks
the 7mm circle to 1.21mm RMSE overall and 0.10mm once settled — essentially at
the simulated sensor noise floor, and matching (not merely approaching) the
MPC controller built on the same M2 model (1.22mm, reproduced independently),
while needing a fraction of the machinery (no Kalman filter, no online
preview/horizon tuning). The gains were picked by sweeping on this exact
task, which the held-out run makes visible: on a 43% larger, 50% faster
circle, RMSE roughly doubles and over a fifth of the commands saturate at the
9 psi rail (the controller is asking for more authority than the actuator
has). That is expected — the whole project's controllers are each tuned on
the scored trajectory — but it means this number should be read as "best
achievable on the task we were asked to track," not as a claim that the
controller is trajectory-independent.

**What was not done here.** The disturbance-observer and Smith-predictor
pieces were not individually ablated in this notebook (see
`scripts/ndosmc.py --sweep` in the source repo for the gain sweep and the
no-observer ablation that motivated the chosen gains). The model itself —
not just the controller — was validated separately via open-loop prediction
against three held-out experiments never used in fitting; that validation is
what justified trusting M2 enough to build this controller on it in the first
place, and is documented alongside the model notes rather than repeated here.